In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


@dataclass(frozen=True)
class Paths:
    data: Path = Path("data") / "data.xlsx"
    output_dir: Path = Path("outputs") / "figure_7"


GASOLINE_95_ROOTS = {
    "SH1", "SH5", "SH9", "SH13", "SH17",
    "T1", "T4", "T7", "T10", "T13",
    "Te1", "Te4", "Te7", "Te10", "Te13",
}

GASOLINE_98_ROOTS = {
    "SH2", "SH6", "SH10", "SH14", "SH18",
    "T2", "T5", "T8", "T11", "T14",
    "Te2", "Te5", "Te8", "Te11", "Te14",
}

GASOLINE_ROOTS = GASOLINE_95_ROOTS | GASOLINE_98_ROOTS


def root_from_id(sample_id: str) -> str:
    return str(sample_id).strip().split("-", 1)[0]


def _sort_spectral_columns(df: pd.DataFrame) -> Tuple[pd.DataFrame, np.ndarray]:
    numeric_cols: List[str] = []
    numeric_axis: List[float] = []

    for col in df.columns:
        try:
            x = float(col)
        except (TypeError, ValueError):
            continue
        numeric_cols.append(col)
        numeric_axis.append(x)

    if not numeric_cols:
        raise ValueError("No numeric spectral columns found. Spectral headers must be numeric wavelengths.")

    axis_arr = np.array(numeric_axis, dtype=float)
    order = np.argsort(axis_arr)
    sorted_cols = [numeric_cols[i] for i in order]
    sorted_axis = axis_arr[order]

    return df[sorted_cols], sorted_axis


def label_from_root(root: str) -> str:
    if root in GASOLINE_95_ROOTS:
        return "#95 Gasoline"
    if root in GASOLINE_98_ROOTS:
        return "#98 Gasoline"
    raise RuntimeError(f"Unknown gasoline root: {root}")


def load_all_gasoline_raw(paths: Paths) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    df = pd.read_excel(paths.data, sheet_name="Positive", index_col=0)
    df = df.sort_index()
    df.index = df.index.to_series().astype(str)

    df_spectral, wavelengths = _sort_spectral_columns(df)

    roots = df_spectral.index.to_series().apply(root_from_id).astype(str)
    keep_mask = roots.isin(GASOLINE_ROOTS)

    if not np.any(keep_mask):
        raise RuntimeError("No #95/#98 gasoline samples were found.")

    X_raw = df_spectral.loc[keep_mask].to_numpy(dtype=float)
    y = np.array([label_from_root(r) for r in roots.loc[keep_mask].to_numpy()], dtype=object)

    return X_raw, y, wavelengths


def plot_raw_mean_spectrum_overlay(
    X_raw: np.ndarray,
    y: np.ndarray,
    wavelengths: np.ndarray,
    output_dir: Path,
) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots(figsize=(8.2, 4.8))

    for cls, color in [("#95 Gasoline", "C0"), ("#98 Gasoline", "C1")]:
        mask = y == cls
        if not np.any(mask):
            continue
        mean_spectrum = np.mean(X_raw[mask, :], axis=0)
        ax.plot(wavelengths, mean_spectrum, linewidth=1.6, color=color, label=cls)

    ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Absorbance (a.u.)")
    ax.set_xlim(1300, 2600)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", which="both", direction="in")

    legend = ax.legend(loc="best", frameon=True)
    legend.get_frame().set_linewidth(0.8)
    legend.get_frame().set_edgecolor("black")

    fig.tight_layout()

    out_path = output_dir / "gasoline_95_98_raw_mean_spectrum_overlay.png"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    return out_path


def run(paths: Paths) -> None:
    X_raw, y, wavelengths = load_all_gasoline_raw(paths)

    plot_path = plot_raw_mean_spectrum_overlay(
        X_raw=X_raw,
        y=y,
        wavelengths=wavelengths,
        output_dir=paths.output_dir,
    )

    print(f"Saved Figure 7: {plot_path}")


if __name__ == "__main__":
    cfg = Paths()
    run(cfg)
